TF-IDF (text only!!) Baseline

In [3]:
#import libraries
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

import seaborn as sns                       #visualisation
%matplotlib inline
sns.set(color_codes=True)
data = pd.read_csv("wine_cleaned.csv")

In [4]:
X = data['Description'] # all descriptions are our "documents" from the 'Description' corpus
y = data['Price']

In [5]:
#test train split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [10]:
#create tfidf features
from sklearn.feature_extraction.text import TfidfVectorizer

TfidfVectorizer(
    stop_words='english',
    max_features=10000,
    ngram_range=(1,2), #consider unigramsand bigrams
    min_df=2, # removes words tha appear in kess than 2 documents
    max_df=0.9 # ignores words that appear in more than 90% of documents
)

tfidf = TfidfVectorizer()

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [11]:
#train random forest regressor
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

model.fit(X_train_tfidf, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsample

In [12]:
#predict
y_pred = model.predict(X_test_tfidf)

In [13]:
#evaluate
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)

MAE: 13.99937645994831
RMSE: 27.04479554361393
R²: 0.17594871709234172


In [14]:
data['Price'].describe()

count    1290.00000
mean       28.58569
std        35.54388
min         4.99000
25%        12.99000
50%        16.99000
75%        29.99000
max       430.00000
Name: Price, dtype: float64

In [15]:
#idf scores - sows word rarity globally
print('\nIDF values:')

for word, score in zip(
    tfidf.get_feature_names_out(),
    tfidf.idf_
):
    print(word, ':', score)

#vocabulary indexes
#print('\nWord indexes:')
#print(tfidf.vocabulary_)



IDF values:
000 : 5.455315819331638
000bc : 7.247075288559693
0345 : 7.247075288559693
10 : 4.572926639133165
100 : 4.251343015005702
1000 : 6.841610180451529
100km : 6.553928107999748
108 : 7.247075288559693
11 : 5.7429978917834195
110 : 6.841610180451529
111 : 7.247075288559693
1146 : 7.247075288559693
11th : 6.553928107999748
12 : 4.443714907653158
120 : 6.148462999891583
1200 : 7.247075288559693
1240 : 7.247075288559693
1250 : 7.247075288559693
1268 : 7.247075288559693
12th : 6.330784556685538
13 : 5.860780927439802
130 : 6.841610180451529
1395 : 7.247075288559693
13th : 5.30116513950438
14 : 5.637637376125593
140 : 6.841610180451529
140ha : 7.247075288559693
1447 : 7.247075288559693
15 : 5.542327196321268
150 : 6.148462999891583
1503 : 6.841610180451529
155 : 7.247075288559693
1565 : 7.247075288559693
1588 : 7.247075288559693
1589 : 7.247075288559693
15th : 6.553928107999748
16 : 5.375273111658102
160 : 6.841610180451529
1607 : 7.247075288559693
1620 : 7.247075288559693
1625 : 6.

In [16]:
#tf-idf matrix       document 0, word 521 → tf-idf score 0.42
print(X_train_tfidf)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 69980 stored elements and shape (1032, 7029)>
  Coords	Values
  (0, 4150)	0.1830305997567525
  (0, 813)	0.11307372646378126
  (0, 3260)	0.12787576047684387
  (0, 75)	0.17533436730981122
  (0, 506)	0.1351948337222095
  (0, 584)	0.058020457177164066
  (0, 4424)	0.056878406129145936
  (0, 4397)	0.08222392034522848
  (0, 6311)	0.056651822461369884
  (0, 4415)	0.10671540088824016
  (0, 4237)	0.11730662395524377
  (0, 1279)	0.1336500069871957
  (0, 6320)	0.06264916307316942
  (0, 6665)	0.09388559237937052
  (0, 5741)	0.08002928776932995
  (0, 1006)	0.1938778301542868
  (0, 4479)	0.10158182482631181
  (0, 6955)	0.1366255776738621
  (0, 6928)	0.0906976069357983
  (0, 2097)	0.1938778301542868
  (0, 2964)	0.1232762856213804
  (0, 4488)	0.08261705308743329
  (0, 3200)	0.17533436730981122
  (0, 1995)	0.24286109911437834
  (0, 5755)	0.13509644367026707
  :	:
  (1031, 1872)	0.1099656062901483
  (1031, 4966)	0.10546758697785664
  (1031, 53